# Collection Operations Report - 数据抽取

**版本**: v2.1  
**更新说明**: 添加 `natural_month_repay` 月度累计回款数据，用于月时序回收进度  
**日期**: 请根据需要修改 `dt_start` 和 `dt_end`

In [1]:
# 参数配置
dt_start = '2026-03-01'  # 开始日期
dt_end = '2026-03-17'    # 结束日期

In [2]:
from get_write_data_from_dataworks import run_sql
import pandas as pd

---

## 1. TL Data - 每日组维度指标

**源表**: `phl_anls.tmp_liujun_ana_11_agent_process_daily`  
**用途**: TL View 的日指标

In [3]:
sql=f'''
-- TL Data: 每日组维度（Group × Day）
-- 用于 TL View 的日指标
SELECT
  owner_bucket AS module,
  owner_group AS group_id,
  dt,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  COUNT(DISTINCT owner_name) AS ownercount,
  -- 在手案量
  SUM(owing_case_cnt) AS total_owing_case,
  -- 覆盖
  SUM(comm_times) AS total_comm_times,
  SUM(own_uncomm_case_cnt) AS total_uncomm_case,
  -- 拨打
  SUM(call_times) AS total_calls,
  SUM(call_connect_times) AS total_connect,
  -- 接通率
  CASE WHEN SUM(call_times) > 0 THEN SUM(call_connect_times) / SUM(call_times) END AS connect_rate
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0  -- 仅内催
  AND owner_bucket IN ('S0','S1','S2','M1','M2')
GROUP BY
  owner_bucket,
  owner_group,
  dt
'''
tl_data=run_sql(sql)
tl_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260318095432521gxtoqgs7hrs5&token=TjlFNnVPUnNmTFRSWGdzZS8yVXhJSWRhSWJFPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NjQxOTY3Mix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMzE4MDk1NDMyNTIxZ3h0b3FnczdocnM1Il19XSwiVmVyc2lvbiI6IjEifQ==


,module,group_id,dt,headcount,ownercount,total_owing_case,total_comm_times,total_uncomm_case,total_calls,total_connect,connect_rate
0,M1,M1-Large A,2026-03-02,11,12,1741,15986,18,16669,497,0.029816
1,M1,M1-Large A,2026-03-03,12,12,1789,12464,125,12893,388,0.030094
2,M1,M1-Large A,2026-03-04,12,12,1787,14996,50,15504,480,0.030960
3,M1,M1-Large A,2026-03-05,10,12,1759,15846,47,16422,499,0.030386
4,M1,M1-Large A,2026-03-06,12,12,1702,12850,62,13361,399,0.029863


---

## 2. STL Data - 每周模块维度指标

**源表**: `phl_anls.tmp_liujun_ana_11_agent_process_daily`  
**用途**: STL View 的周指标

In [4]:
sql=f'''
-- STL Data: 每周模块维度（Module × Week）
-- 用于 STL View 的周指标
SELECT
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END AS module,
  -- 周区间：周起始为周六
  CONCAT(
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), -1, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), - (WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 1), 'dd'), 'yyyy-MM-dd')
    END,
    '-',
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')), 'dd'), 'yyyy-MM-dd')
    END
  ) AS week,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  -- 在手案量
  SUM(owing_case_cnt) AS total_owing_case,
  -- 覆盖
  SUM(comm_times) AS total_comm_times,
  -- 拨打
  SUM(call_times) AS total_calls,
  SUM(call_connect_times) AS total_connect
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1','M2')
GROUP BY
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END,
  CONCAT(
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), -1, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), - (WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 1), 'dd'), 'yyyy-MM-dd')
    END,
    '-',
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')), 'dd'), 'yyyy-MM-dd')
    END
  )
'''
stl_data=run_sql(sql)
stl_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260318095434625gql06misbr&token=eVlhalNVZ0VFUkFRZ29sZ1MxR3pyT0VIYlg4PSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NjQxOTY3NCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMzE4MDk1NDM0NjI1Z3FsMDZtaXNiciJdfV0sIlZlcnNpb24iOiIxIn0=


,module,week,headcount,total_owing_case,total_comm_times,total_calls,total_connect
0,M1,2026-03-01-2026-03-07,31,32931,231656,235518,6614
1,M1,2026-03-08-2026-03-14,31,32860,252867,257416,7119
2,M1,2026-03-15-2026-03-21,31,13024,83457,84680,2535
3,M2,2026-03-01-2026-03-07,4,16726,27188,27337,924
4,M2,2026-03-08-2026-03-14,4,18010,31101,31212,1048


---

## 3. Agent Performance - Agent日明细

**源表**: `phl_anls.tmp_liujun_ana_11_agent_process_daily`  
**用途**: TL View drill-down

In [5]:
sql=f'''
-- Agent Performance: Agent日明细（Group × Agent × Day）
-- 用于 TL View  drill-down
SELECT
  owner_bucket AS module,
  owner_group AS group_id,
  owner_name AS agent_id,
  dt,
  -- 出勤
  last_call_hour - first_call_hour AS work_hours,
  CASE WHEN last_call_hour - first_call_hour >= 8 THEN 1 ELSE 0 END AS is_full_attendance,
  -- 在手案量
  owing_case_cnt AS owing_case,
  -- 覆盖
  comm_times,
  own_uncomm_case_cnt AS uncomm_case,
  -- 拨打
  call_times,
  call_connect_times AS connect_times,
  -- 接通率
  CASE WHEN call_times > 0 THEN call_connect_times / call_times END AS connect_rate
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1','M2')
ORDER BY
  owner_bucket, owner_group, owner_name, dt
'''
agent_performance=run_sql(sql)
agent_performance.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=202603180954362g2m06misbr&token=RU9WOUtPenIvYWR3MW53ZWRKaW5TdHNmMFJjPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NjQxOTY3Nix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMzE4MDk1NDM2MmcybTA2bWlzYnIiXX1dLCJWZXJzaW9uIjoiMSJ9


,module,group_id,agent_id,dt,work_hours,is_full_attendance,owing_case,comm_times,uncomm_case,call_times,connect_times,connect_rate
0,M1,M1-Large A,Ramos02,2026-03-02,9.983333,1,1,4,0,4,0,0.000000
1,M1,M1-Large A,Ramos02,2026-03-02,9.983333,1,1,3,0,3,0,0.000000
2,M1,M1-Large A,Ramos02,2026-03-02,9.983333,1,84,595,0,595,19,0.031933
3,M1,M1-Large A,Ramos02,2026-03-02,9.983333,1,60,741,0,741,18,0.024291
4,M1,M1-Large A,Ramos02,2026-03-03,10.150000,1,1,0,1,0,0,NaN


---

## 4. Group Performance - 组周明细

**源表**: `phl_anls.tmp_liujun_ana_11_agent_process_daily`  
**用途**: STL View drill-down

In [6]:
sql=f'''
-- Group Performance: 组周明细（Module × Group × Week）
-- 用于 STL View drill-down
SELECT
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END AS module,
  owner_group AS group_id,
  CONCAT(
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), -1, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), - (WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 1), 'dd'), 'yyyy-MM-dd')
    END,
    '-',
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')), 'dd'), 'yyyy-MM-dd')
    END
  ) AS week,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  -- 在手案量
  SUM(owing_case_cnt) AS total_owing_case,
  -- 覆盖
  SUM(comm_times) AS total_comm_times,
  SUM(own_uncomm_case_cnt) AS total_uncomm_case,
  -- 拨打
  SUM(call_times) AS total_calls,
  SUM(call_connect_times) AS total_connect
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1','M2','T2','T4','T5')
GROUP BY
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END,
  owner_group,
  CONCAT(
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), -1, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), - (WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 1), 'dd'), 'yyyy-MM-dd')
    END,
    '-',
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')), 'dd'), 'yyyy-MM-dd')
    END
  )
'''
group_performance=run_sql(sql)
group_performance.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260318095437470gbm06misbr&token=STlwd0UxTkVONm1EVktReENYT3VIeGl0Qm80PSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NjQxOTY3Nyx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMzE4MDk1NDM3NDcwZ2JtMDZtaXNiciJdfV0sIlZlcnNpb24iOiIxIn0=


,module,group_id,week,headcount,total_owing_case,total_comm_times,total_uncomm_case,total_calls,total_connect
0,M1,M1-Large A,2026-03-01-2026-03-07,12,10478,86514,340,89790,2713
1,M1,M1-Large A,2026-03-08-2026-03-14,12,9718,94505,180,98291,2798
2,M1,M1-Large A,2026-03-15-2026-03-21,12,3907,32151,116,33160,988
3,M1,M1-Large B,2026-03-01-2026-03-07,12,10020,97171,492,97452,2029
4,M1,M1-Large B,2026-03-08-2026-03-14,12,9716,104119,151,104417,2351


---

## 5. 日目标回款 + Achievement（日粒度）

**源表**: `phl_anls.rpt_coll_repay_target_dly4`  
**用途**: Module Daily Trends、Risk Analysis、Achievement 计算  
**说明**: 此表已包含日粒度的目标回款数据（target_repay_principal），可直接计算 achievement

In [7]:
sql=f'''
-- 日目标回款 + Achievement
-- 源表: phl_anls.rpt_coll_repay_target_dly4
-- 用途: Module Daily Trends、Risk Analysis
SELECT
  TO_CHAR(dt, 'yyyyMM') AS month,
  TO_CHAR(dt, 'yyyy-MM-dd') AS dt,
  owner_bucket,
  owner_group,
  SUM(owing_principal) AS daily_owing_principal,
  SUM(repay_principal) / NULLIF(SUM(owing_principal), 0) AS daily_repay_rate,
  SUM(target_repay_principal) AS target_repay_principal,
  SUM(repay_principal) AS actual_repay_principal,
  CASE WHEN SUM(target_repay_principal) <= 0 THEN 0
       ELSE SUM(repay_principal) / SUM(target_repay_principal)
  END AS achieve_rate
FROM (
  SELECT
    *,
    CASE
      WHEN owner_bucket = 'S0' AND case_overdue_days = -3 THEN 'H3'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -2 THEN 'H2'
      WHEN owner_bucket = 'S0' AND case_overdue_days = -1 THEN 'H1'
      WHEN owner_bucket = 'S0' AND case_overdue_days >= 0 THEN 'H0'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 1 THEN '<=1'
      WHEN owner_bucket = 'S1' AND case_overdue_days <= 5 THEN '2-5'
      WHEN owner_bucket = 'S1' AND case_overdue_days >= 6 THEN '6-7'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 8 THEN '<=8'
      WHEN owner_bucket = 'S2' AND case_overdue_days <= 12 THEN '9-12'
      WHEN owner_bucket = 'S2' AND case_overdue_days >= 13 THEN '13-15'
      WHEN owner_bucket = 'M1' AND case_overdue_days <= 16 THEN '<=16'
      WHEN owner_bucket = 'M1' AND case_overdue_days >= 17 THEN '17-25'
      WHEN owner_bucket = 'M1' THEN '26-30'
      ELSE case_overdue_days
    END AS case_stage,
    CASE
      WHEN case_owing_principal <= 1000 THEN '<=1000'
      WHEN case_owing_principal <= 2000 THEN '1000-2000'
      WHEN case_owing_principal <= 5000 THEN '2000-5000'
      WHEN case_owing_principal > 5000 THEN '>5000'
    END AS principal_stage
  FROM phl_anls.rpt_coll_repay_target_dly4
  WHERE dt >= '{dt_start}'
    AND case_overdue_days > -99
    AND owner_bucket IN ('S0', 'S1', 'S2', 'M1')
) t
WHERE owner_bucket IN ('M1', 'S1', 'S2', 'S0')
  AND TO_CHAR(dt, 'yyyyMMdd') BETWEEN REPLACE('{dt_start}', '-', '') AND REPLACE('{dt_end}', '-', '')
GROUP BY
  TO_CHAR(dt, 'yyyyMM'),
  TO_CHAR(dt, 'yyyy-MM-dd'),
  owner_bucket,
  owner_group
HAVING SUM(owing_principal) > 0
ORDER BY month DESC, dt DESC, owner_bucket DESC, owner_group ASC
'''
daily_target_repay=run_sql(sql)
daily_target_repay.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260318095439392gom06misbr&token=b2tuWGh6RS9hS1owZXlnZUYySmZweXFXeGdJPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NjQxOTY3OSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMzE4MDk1NDM5MzkyZ29tMDZtaXNiciJdfV0sIlZlcnNpb24iOiIxIn0=


,month,dt,owner_bucket,owner_group,daily_owing_principal,daily_repay_rate,target_repay_principal,actual_repay_principal,achieve_rate
0,202603,2026-03-17,S2,S2-A-AJCAI,4100861.2,0.01813570281286282,103790.601412542226706702,74372,0.716558137132181271
1,202603,2026-03-17,S2,S2-C-MBA,4143582.7,0.012999378532978236,94208.233850818493361677,53864,0.571754694874072545
2,202603,2026-03-17,S2,S2-Large A,3725992.92,0.011272431510685748,95117.72338373893345898,42001,0.441568600528346712
3,202603,2026-03-17,S2,S2-Large B,3637926.01,0.018437153426328206,85316.254825622266601165,67073,0.786169061652909859
4,202603,2026-03-17,S2,S2-Large C,3559845.01,0.019237916203548424,93277.107362108400077721,68484,0.734199440106351234


---

In [15]:
# Natural Month Repay: 月度累计回款
# dt 为业务后一天，因此 substr(dt,9,2)='01' 表示"上月月末累计状态"
sql=f'''
SELECT
  SUBSTR(dt_biz, 1, 7) AS month,
  dt_biz,
  agent_bucket AS module,
  group_name,
  SUM(repay_principal) AS repay_principal,
  SUM(start_owing_principal) AS start_owing_principal,
  CASE WHEN SUM(start_owing_principal) > 0 
       THEN SUM(repay_principal) / SUM(start_owing_principal) 
  END AS repay_rate
FROM tmp_maoruochen_phl_repay_natural_day_daily
WHERE dt_biz >= '{dt_start}'
  AND dt_biz <= '{dt_end}'
  AND in_or_out = 'inhouse'             -- 仅内催
  AND data_level = '4.组别层级'         -- 模块维度
  AND case_bucket IN ('S0', 'S1', 'S2', 'M1')
--   AND case_bucket IS NOT NULL
GROUP BY
  SUBSTR(dt_biz, 1, 7),
  dt_biz,
  agent_bucket,
  group_name
ORDER BY month DESC, module DESC
'''
natural_month_repay=run_sql(sql)
natural_month_repay.head(10)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260318110250855g1y3nrz1al7&token=NXRzSGZ2emxnMlNMMlh1dWVGNllxanh2NEw4PSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NjQyMzc3MCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMzE4MTEwMjUwODU1ZzF5M25yejFhbDciXX1dLCJWZXJzaW9uIjoiMSJ9


,month,dt_biz,module,group_name,repay_principal,start_owing_principal,repay_rate
0,2026-03,2026-03-07,S2_Small,S2-Small A,198560.54,1626956.46,0.122044163369928166
1,2026-03,2026-03-06,S2_Small,S2-Small A,169811.46,1486700.46,0.114220358820632907
2,2026-03,2026-03-12,S2_Small,S2-Small A,368753.54,2847233.02,0.129512947275386684
3,2026-03,2026-03-05,S2_Small,S2-Small A,125254.96,1358722.21,0.092185848643778333
4,2026-03,2026-03-16,S2_Small,S2-Small A,545687.54,3536066.98,0.15432047613532479
5,2026-03,2026-03-08,S2_Small,S2-Small A,215900.54,2112432.98,0.102204681542133469
6,2026-03,2026-03-15,S2_Small,S2-Small A,492714.54,3336615.02,0.147668981002189458
7,2026-03,2026-03-03,S2_Small,S2-Small A,80416.71,1087451.49,0.07394969866655845
8,2026-03,2026-03-14,S2_Small,S2-Small A,468905.54,3183797.02,0.147278716907650099
9,2026-03,2026-03-10,S2_Small,S2-Small A,297582.54,2514505.98,0.11834632423502926


---

## 6. Natural Month Repay - 月度累计回款

**源表**: `tmp_maoruochen_phl_repay_natural_day_daily`  
**用途**: 月时序回收进度（Month Trends）  
**说明**: `dt` 为业务后一天，`substr(dt,9,2)='01'` 表示"上月月末累计状态"

## 输出说明

| Cell | 输出变量名 | 用途 |
|------|-----------|------|
| 1 | `tl_data` | TL每日指标 |
| 2 | `stl_data` | STL每周指标 |
| 3 | `agent_performance` | Agent明细 |
| 4 | `group_performance` | 组周明细 |
| 5 | `daily_target_repay` | 日目标回款（含achievement） |
| 6 | `natural_month_repay` | 月度累计回款（Month Trends） |

In [16]:
# 输出为 Excel（推荐）
with pd.ExcelWriter('260318_output_automation_v2.xlsx') as writer:
    tl_data.to_excel(writer, sheet_name='tl_data', index=True)
    stl_data.to_excel(writer, sheet_name='stl_data', index=True)
    agent_performance.to_excel(writer, sheet_name='agent_performance', index=True)
    group_performance.to_excel(writer, sheet_name='group_performance', index=True)
    daily_target_repay.to_excel(writer, sheet_name='daily_target_repay', index=True)
    natural_month_repay.to_excel(writer, sheet_name='natural_month_repay', index=True)

In [10]:
# 输出为 CSV（可选）
# tl_data.to_csv('tl_data.csv', index=False)
# stl_data.to_csv('stl_data.csv', index=False)
# agent_performance.to_csv('agent_performance.csv', index=False)
# group_performance.to_csv('group_performance.csv', index=False)
# daily_target_repay.to_csv('daily_target_repay.csv', index=False)
# natural_month_repay.to_csv('natural_month_repay.csv', index=False)